In [ ]:
pip install transformers scikit-learn torch numpy pandas peft ollama triton bitsandbytes

In [1]:
#CLEAR DISK SPACE AND CACHE

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
torch.cuda.empty_cache()
#torch.cuda.reset_peak_memory_stats()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [2]:
#IMPORT PACKAGES AND MODEL

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import pandas as pd
import numpy as np
import torch
import json

model_name = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name, 
                                             device_map = "auto",
                                             )

pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype="auto",
    device_map = "auto"
)



2025-08-18 19:08:18.962643: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-18 19:08:18.975362: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755558498.990009 1283948 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755558498.994657 1283948 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755558499.006722 1283948 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:

#LOAD DATA AND PACKAGES

#import packages and datasets
import pandas as pd
import numpy as np
import sklearn as sk
import json
from pathlib import Path
from typing import Sequence
import re
import os

#Unlabelled dataset
df_test = pd.read_csv("unlabelled_frames_test.csv")
#Labelled dataset
df_train = pd.read_csv('labelled_frames_train.csv')
df_test_lab = pd.read_csv('labelled_frames_test.csv')
df_gold = pd.concat([df_train, df_test_lab], axis=0).reset_index(drop=True)
#Frame names
frame_names = df_gold.columns[7:14].to_list()
#Codebook
with open('framing_codebook_short.json', 'r') as f:
    codebook = json.load(f)
codebook_str = json.dumps(codebook, indent=2, ensure_ascii=False)



In [3]:
#HELPER FUNCTIONS

##parsing responses from model
def parse_json_with_fallback(content, frame_names):
        #if the response can be parsed
        dict_val = json.loads(content)
        #remove any leading or trailing white spaces in the keys
        cleaned_dict = {k.strip(): v for k, v in dict_val.items()}

        annotations = []

        #parse response for each frame
        for f in frame_names:
            try:
                f_val = str(cleaned_dict.get(f))

                binary_val = pd.NA
                if re.search("yes",f_val.strip().lower()):
                    binary_val = 1
                if re.search("no",f_val.strip().lower()):
                    binary_val = 0

            except:
                binary_val = pd.NA
            
            annotations.append(binary_val)

        return annotations



##compute kappa and accuracy
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner', suffixes = ('_pred', '_gold'))   # keep only matching IDs
            .dropna(subset=[column_pred + '_pred', column_gold + '_pred'])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred + '_pred'] = df_temp[column_pred + '_pred'].astype(int)
    df_temp[column_gold + '_gold'] = df_temp[column_gold + '_gold'].astype(int)

    #compute scores
    acc = round(100*sk.metrics.accuracy_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    f1 = round(sk.metrics.f1_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    precision = round(sk.metrics.precision_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    recall = round(sk.metrics.recall_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)

    print("Accuracy for " + column_gold +  " is " + str(acc) + "%, and Cohen's kappa is " + str(k))

    return([acc, k, f1, precision, recall])


In [ ]:
#ZERO SHOT: ONE-TO-MANY

#ONE-TO-MANY PROMPTS
SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 

Provide your response in a JSON array format, as follows, and include NOTHING ELSE in the response: 
{"sexual stigma and transmission routes": "yes/no",
"racial disparities and stigmatising name": "yes/no",
"global relations": "yes/no",
"public health failure": "yes/no",
"epidemic preparedness and surveillance": "yes/no",
"human-interest stories": "yes/no",
"broader health issues": "yes/no"}.

Identify the frames from the codebook that are present in the article. Here is the codebook with frame definitions: """ + codebook_str


USER_PROMPT =  """Identify frames in the article with the following guidelines:
1. Read the entire article carefully before coding
2. Identify all frames present in each article
3. Some articles maybe irrelevant to the Mpox epidemic or may not contain any of the frames mentioned in the codebook. In this case, mark "no" for every frame. 
4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
5. Use frame descriptions and examples to guide decision
6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no". """

#sample for testing things first
#df_sample = df_test.sample(n= 5, random_state= 42).reset_index(drop=True)

#otherwise
df_sample = df_test.copy()

df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = pipeline(
        messages,
        max_new_tokens=2048)

        predictions.append(outputs[0]["generated_text"][-1])
    
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save responses
with open("Predicted/Test/gpt-oss-20b-z_shot_predictions.txt", "w") as file:
    for item in predictions:
        file.write(f"{item}\n")


In [ ]:
#PARSE GPT RESPONSES AND CONVERT TO WHAT WE NEED

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import pandas as pd
import numpy as np
import torch
import json

#load the previous output
with open("Predicted/Test/gpt-oss-20b-z_shot_predictions.txt", 'r') as file:
    predictions = file.readlines()
    

model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

#8-bit quantise
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,               
    llm_int8_threshold=6.0,           
    llm_int8_enable_fp32_cpu_offload=True,
    llm_int8_skip_modules=["output_layer"]
)

model = AutoModelForCausalLM.from_pretrained(model_name, 
                                             quantization_config=bnb_config, #if quantizing
                                             device_map = "auto",
                                             #cache_dir = "/tmp"
                                             )

pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    #torch_dtype="auto", #if not quantising
    device_map = "auto"
)

#PROMPTS
SYSTEM_PROMPT = """You're a simple NLP tool that parses another models response and outputs in a specified format.
An LLM performed a content analytic task and output a string as a response with answers and justifications. 
Your job is to parse the response and just output the answers without justification. 

Provide your response in a JSON format, as follows, and include NOTHING ELSE in the response: 
{"sexual stigma and transmission routes": "yes/no",
"racial disparities and stigmatising name": "yes/no",
"global relations": "yes/no",
"public health failure": "yes/no",
"epidemic preparedness and surveillance": "yes/no",
"human-interest stories": "yes/no",
"broader health issues": "yes/no"}

"""


USER_PROMPT =  """Note: Sometimes the answer may be incomplete from the LLM's end. In that case, just output NA. 
Parse the response and output the answers in correct format for the following LLM response: """

answers = []

#now parse through the output
for i in range(len(predictions)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + predictions[i] 
        }

            ]
    
        outputs = pipeline(
        messages,
        max_new_tokens=2048)

        answers.append(outputs[0]["generated_text"][-1])
    
    
    #if there is an error
    except Exception as e:
            answers.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save the responses
df1 = df_train.copy()
df1[frame_names] = pd.NA

for j in range(len(answers)):
    if answers[j] == None:
        df1.loc[j, frame_names] = pd.NA

    else:
        content = answers[j].get('content')
        content = content.replace("[", "").replace("]", "")

        try: 
            annotation = parse_json_with_fallback(content, frame_names)
            df1.loc[j, frame_names] = annotation


        #or maybe not
        except Exception as e:
            df1.loc[j, frame_names] = pd.NA


#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1', 'Precision', 'Recall'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    
df1.to_csv('Predicted/Test/gpt-oss-20b-zshot.csv')

2025-08-18 19:34:29.052751: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-18 19:34:29.065549: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755560069.080720 1412273 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755560069.085469 1412273 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755560069.097519 1412273 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


0 iterations finished


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

380 iterations finished


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


390 iterations finished


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Accuracy for sexual stigma and transmission routes is 60.15%, and Cohen's kappa is 0.25
Accuracy for racial disparities and stigmatising name is 87.469%, and Cohen's kappa is 0.255
Accuracy for global relations is 66.165%, and Cohen's kappa is 0.092
Accuracy for public health failure is 73.935%, and Cohen's kappa is 0.211
Accuracy for epidemic preparedness and surveillance is 52.882%, and Cohen's kappa is 0.128
Accuracy for human-interest stories is 87.469%, and Cohen's kappa is 0.327
Accuracy for broader health issues is 59.649%, and Cohen's kappa is 0.184
